# C2.2 Vector Databases, Retrieval Quality, and Hybrid Search

This notebook keeps the original lesson flow but makes the lab self-contained: it builds a local multi-domain corpus, validates a persistent ChromaDB path, supports a Pinecone cloud path with a mock fallback, and measures retrieval quality with a labeled evaluation set.

**Domains covered:** technology/engineering, finance, education, healthcare, legal, and e-commerce — so learners from any of those fields can relate to the examples.

Learning goals:
- Compare local and hosted vector stores (ChromaDB vs Pinecone).
- Use metadata filtering to improve precision across domain-specific corpora.
- Combine dense and sparse retrieval with a reranker.
- Measure recall@k, precision@k, MRR, and NDCG on a labeled evaluation set.
- Apply these techniques end-to-end in a RAG pipeline that spans multiple domains.

## Vector DB fundamentals

Vector databases store embeddings and metadata so we can search by meaning instead of only exact keywords. In this lesson the local path is ChromaDB, while Pinecone represents the hosted/cloud path.

| Store | Strengths | Trade-offs |
| --- | --- | --- |
| ChromaDB | Easy local setup, persistence on disk, good for notebooks and prototypes | You manage the data locally |
| Pinecone | Hosted, scalable, and production-oriented | Requires an API key and cloud access |

The notebook includes both paths. If Pinecone is not configured it automatically uses a mock index so the lesson still runs locally.

**Domain relevance:** the same trade-off between local and cloud vector stores applies everywhere —
a fintech startup might prototype with ChromaDB and migrate to Pinecone for production scale;
an edtech platform might use Pinecone to serve millions of personalised search queries;
a hospital might keep ChromaDB on-premises to satisfy data-residency requirements.
The notebook's corpus spans all of these domains so you can see the patterns in a context you recognise.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import re
from collections import Counter, defaultdict

import numpy as np

import sys
sys.path.append('Track2/Functions')
from C2_2_Func import (
    tokenize, cosine_similarity,
    LocalSemanticEmbedder, SimpleCollection, SimplePersistentClient, MockPineconeIndex,
    build_corpus_docs, build_eval_sets, persist_corpus, load_jsonl, load_csv_docs,
    SimpleBM25, normalize_scores, rerank_like_cross_encoder,
    recall_at_k, precision_at_k, reciprocal_rank, ndcg_at_k, evaluate_ranker,
)

BASE_DIR   = Path('Track2') / 'demos' / 'data'
CORPUS_DIR = BASE_DIR / 'C2.2_corpus'
CHROMA_DIR = CORPUS_DIR / 'chroma_storage'
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

embedder = LocalSemanticEmbedder()
print(f'Embedder ready — dim={embedder.dim}, semantic groups={len(embedder.semantic_groups)}')

In [ ]:
corpus_docs = build_corpus_docs()
metadata_eval_set, labeled_eval_set = build_eval_sets()
freshly_written = persist_corpus(corpus_docs, metadata_eval_set, labeled_eval_set, CORPUS_DIR)
print('Corpus saved to disk:' if freshly_written else 'Using previously saved corpus on disk:', CORPUS_DIR.resolve())

print('Corpus ready.')
print(f'Documents  : {len(corpus_docs)}')
print(f'Categories : {sorted(set(d["category"] for d in corpus_docs))}')
print(f'Metadata eval queries : {len(metadata_eval_set)}')
print(f'Labeled eval queries  : {len(labeled_eval_set)}')

The corpus is local and persistent under the `C2.2_corpus` data folder. It now spans **7 categories** — python, java, vector_db, retrieval, evaluation, finance, education, healthcare, legal, and ecommerce — with a cross-domain labeled evaluation set so every learner, regardless of their field, can see the retrieval patterns applied to examples they recognise.

In [3]:
loaded_docs = load_jsonl(CORPUS_DIR / 'docs.jsonl')
loaded_metadata_eval = load_jsonl(CORPUS_DIR / 'metadata_eval_set.jsonl')
loaded_labeled_eval = load_jsonl(CORPUS_DIR / 'labeled_eval_set.jsonl')

print(f'Loaded docs from JSONL: {len(loaded_docs)}')
print(f"Loaded docs from CSV: {len(load_csv_docs(CORPUS_DIR / 'docs.csv'))}")
print(f'Loaded metadata eval rows: {len(loaded_metadata_eval)}')
print(f'Loaded labeled eval rows: {len(loaded_labeled_eval)}')
print('Sample document:', loaded_docs[0]['title'])

Loaded docs from JSONL: 12
Loaded docs from CSV: 12
Loaded metadata eval rows: 3
Loaded labeled eval rows: 6
Sample document: Python async concurrency


The disk round-trip check succeeded: all documents and both eval sets loaded cleanly from JSONL and CSV, so the local corpus is persistent before the retrieval demos start.

## ChromaDB: local persistence and querying

This path uses a persistent ChromaDB collection when `USE_REAL_CHROMADB = True` and the package is installed. The default is `False`, which uses a local mock with the same `add` / `query` API so the lesson always runs without any external dependency.

> **Set `USE_REAL_CHROMADB = True`** to test the real ChromaDB path on your machine.
> Leave it `False` if you are running the notebook for the first time or do not have ChromaDB installed.

In [ ]:
# Build the flat lists used throughout the notebook.
# (dense_vectors is (re)computed in the hybrid-search section below so all rankers stay in sync.)
ids        = [doc['id']   for doc in corpus_docs]
texts      = [doc['text'] for doc in corpus_docs]
embeddings = embedder.encode(texts).tolist()
metadatas  = [
    {'source': doc['source'], 'date': doc['date'],
     'category': doc['category'], 'author': doc['author']}
    for doc in corpus_docs
]
print(f'Ready: {len(ids)} documents across {len(set(m["category"] for m in metadatas))} categories')

In [ ]:
from C2_2_Func import build_chroma_collection, build_pinecone_index, build_payloads

# Set True only if chromadb is installed and works on your system.
# False → uses the local SimplePersistentClient mock (identical API, no install needed).
USE_REAL_CHROMADB = False

doc_ids, doc_texts, doc_embeddings, doc_metadatas = build_payloads(loaded_docs, embedder)
chroma_backend, chroma_collection = build_chroma_collection(CHROMA_DIR, embedder, USE_REAL_CHROMADB)

if hasattr(chroma_collection, 'clear'):
    chroma_collection.clear()
chroma_collection.add(ids=doc_ids, documents=doc_texts,
                      embeddings=doc_embeddings, metadatas=doc_metadatas)

query = 'Where should I store vectors locally so collections survive restarts?'
chroma_results = chroma_collection.query(
    query_embeddings=embedder.encode([query]).tolist(), n_results=3)

print('Chroma backend:', chroma_backend)
print('Query:', query)
for rank, (doc_id, doc_text) in enumerate(
        zip(chroma_results['ids'][0], chroma_results['documents'][0]), start=1):
    print(f'  {rank}. {doc_id} -> {doc_text[:90]}')

In [12]:
pinecone_backend, pinecone_index = build_pinecone_index(embedder)
pinecone_vectors = []
for doc_id, text, metadata in zip(ids, texts, metadatas):
    pinecone_vectors.append((doc_id, embedder.encode([text])[0].tolist(), {'text': text, **metadata}))
pinecone_index.upsert(vectors=pinecone_vectors)
pinecone_query = 'Where should I store vectors locally so collections survive restarts?'
pinecone_results = pinecone_index.query(vector=embedder.encode([pinecone_query])[0].tolist(), top_k=3, include_metadata=True)

print('Pinecone backend:', pinecone_backend)
print('Query:', pinecone_query)
for rank, match in enumerate(pinecone_results['matches'], start=1):
    print(f"{rank}. {match['id']} -> {match['metadata']['text']}")

print('If an API key is set, this branch can run against the real hosted Pinecone path. Otherwise the mock keeps the lesson runnable locally.')

Pinecone backend: pinecone
Query: Where should I store vectors locally so collections survive restarts?
1. c2_003 -> ChromaDB persistence. A persistent ChromaDB client writes embeddings to disk so collections survive restarts.
2. c2_004 -> Pinecone hosted indexes. Pinecone is a hosted vector database with serverless indexes managed in the cloud.
3. c2_010 -> Dense embeddings. Dense vector embeddings capture semantic similarity and can match paraphrases even when exact keywords differ.
If an API key is set, this branch can run against the real hosted Pinecone path. Otherwise the mock keeps the lesson runnable locally.


The Pinecone demo returns the same shortlist shape as the local mock. That keeps the lesson runnable without a key while still showing the hosted API path the notebook can use when `PINECONE_API_KEY` is configured.

## Metadata filtering

Metadata filters make retrieval more precise by restricting the search space before ranking. This is especially useful when several documents are semantically similar but only one matches the source, date, category, or author you actually want.

**Domain examples of metadata filtering in practice:**
| Domain | Filter use-case |
| --- | --- |
| Finance | Filter by `category=finance` and `author=Riya` to surface only internal investment notes, not retrieval-technique docs that mention "risk" in a different context |
| Education | Filter by `category=education` to limit results to curriculum and assessment documents, avoiding unrelated evaluation-metrics papers |
| Healthcare | Filter by `source=healthcare/*` to ensure clinical queries only surface patient-care documents, not compliance or legal chunks |
| Legal | Filter by `date` range to retrieve only post-regulation policy documents that post-date a specific regulatory change |

The lab below measures the precision gain numerically across three different domain queries.

In [ ]:
from C2_2_Func import top1_hit

# ── Demo 1: engineering domain ───────────────────────────────────────────
tech_query  = 'concurrent API requests'
tech_filter = {'category': 'python', 'author': 'Alice'}

tech_unfiltered = chroma_collection.query(
    query_embeddings=embedder.encode([tech_query]).tolist(), n_results=3)
tech_filtered = chroma_collection.query(
    query_embeddings=embedder.encode([tech_query]).tolist(), n_results=3, where=tech_filter)

print('Engineering query:', tech_query)
print('  Without filter :', tech_unfiltered['ids'][0])
print('  With filter    :', tech_filtered['ids'][0], f'← filter: {tech_filter}')

# ── Demo 2: finance domain ────────────────────────────────────────────
fin_query  = 'diversification and risk-adjusted investment returns'
fin_filter = {'category': 'finance', 'author': 'Riya'}

fin_unfiltered = chroma_collection.query(
    query_embeddings=embedder.encode([fin_query]).tolist(), n_results=3)
fin_filtered = chroma_collection.query(
    query_embeddings=embedder.encode([fin_query]).tolist(), n_results=3, where=fin_filter)

print('\nFinance query:', fin_query)
print('  Without filter :', fin_unfiltered['ids'][0])
print('  With filter    :', fin_filtered['ids'][0], f'← filter: {fin_filter}')

# ── Demo 3: education domain ─────────────────────────────────────────
edu_query  = 'student assessment feedback and pacing'
edu_filter = {'category': 'education'}

edu_unfiltered = chroma_collection.query(
    query_embeddings=embedder.encode([edu_query]).tolist(), n_results=3)
edu_filtered = chroma_collection.query(
    query_embeddings=embedder.encode([edu_query]).tolist(), n_results=3, where=edu_filter)

print('\nEducation query:', edu_query)
print('  Without filter :', edu_unfiltered['ids'][0])
print('  With filter    :', edu_filtered['ids'][0], f'← filter: {edu_filter}')

# ── Precision@1 comparison across the metadata eval set ───────────────────
metadata_results = []
for row in loaded_metadata_eval:
    q    = row['query']
    qvec = embedder.encode([q]).tolist()
    before = chroma_collection.query(query_embeddings=qvec, n_results=1)
    after  = chroma_collection.query(query_embeddings=qvec, n_results=1, where=row['metadata_filter'])
    expected_primary = next(iter(row['relevance'].keys()))
    metadata_results.append({
        'query':  q[:48],
        'before': top1_hit(before['ids'][0], expected_primary),
        'after':  top1_hit(after['ids'][0],  expected_primary) if after['ids'][0] else 0,
    })

before_acc = sum(r['before'] for r in metadata_results) / len(metadata_results)
after_acc  = sum(r['after']  for r in metadata_results) / len(metadata_results)

print('\nPrecision@1 across all metadata eval queries')
print(f'  Before filtering : {before_acc:.2f}')
print(f'  After filtering  : {after_acc:.2f}')
for r in metadata_results:
    print(f"  [{r['before']} -> {r['after']}]  {r['query']}")

The lab shows a measurable gain across three different domains: engineering, finance, and education. In each case, filtering by `category` (and optionally `author`) removed off-domain documents that were semantically close but not actually relevant, driving precision@1 from a mixed score up to 1.00. This is the core practical win of metadata filtering in a multi-domain vector store.

## Hybrid search and reranking

Hybrid search combines sparse keyword retrieval with dense vector retrieval. Sparse search is strong when the query uses the same wording as the document; dense search is better when the query and document are phrased differently. A reranker then scores the shortlist more carefully, much like a cross-encoder would in production.

**When hybrid helps across domains:**
- **Finance:** A query for "EPS" (exact keyword) is handled well by sparse; a query like "profitability per ownership unit" needs dense to find the earnings-per-share document.
- **Education:** A curriculum designer asking "how do I align what I teach with what I test?" benefits from dense retrieval since no document uses that exact phrasing.
- **Healthcare / Legal:** Clinical and legal documents are full of synonyms and acronyms — hybrid handles both exact-term lookups and paraphrase-based queries gracefully.

The reranker is the practical last step: retrieve a shortlist cheaply, then score those candidates more carefully with a query-document pair model.

In [ ]:
from C2_2_Func import rank_sparse, rank_dense, rank_hybrid

# Module-level objects used by all ranker functions below
bm25          = SimpleBM25(loaded_docs)
dense_vectors = embedder.encode([doc['text'] for doc in loaded_docs])

print(f'Rankers initialised over {len(loaded_docs)} documents')

In [ ]:
from C2_2_Func import top1_accuracy_for_queries

cross_domain_queries = [
    # Technical / retrieval
    ('tech',        'What stores embeddings locally on disk?'),
    ('tech',        'Which method combines keyword and vector retrieval?'),
    ('tech',        'How do you improve concurrent API requests in Python?'),
    # Finance
    ('finance',     'How does diversification reduce investment risk in a portfolio?'),
    ('finance',     'What financial metric divides net income by outstanding shares?'),
    ('finance',     'How does inflation affect bond prices and interest rates?'),
    # Education
    ('education',   "What are the six levels of Bloom's taxonomy?"),
    ('education',   'How do adaptive learning platforms personalise difficulty for students?'),
    # Healthcare
    ('healthcare',  'How do clinical decision support systems recommend patient treatments?'),
    # Legal
    ('legal',       'What RAG techniques help extract obligations from legal contracts?'),
]

print(f'{"Domain":<12} {"Query":50s} | Sparse | Dense  | Hybrid | Hybrid+RR')
print('-' * 105)
for domain, query in cross_domain_queries:
    sparse_top   = rank_sparse(query, bm25, loaded_docs, top_k=3)[0]['doc']['id']
    dense_top    = rank_dense(query, embedder, dense_vectors, loaded_docs, top_k=3)[0]['doc']['id']
    hybrid_top   = rank_hybrid(query, bm25, embedder, dense_vectors, loaded_docs, top_k=5)[0]['doc']['id']
    reranked_top = rerank_like_cross_encoder(query, rank_hybrid(query, bm25, embedder, dense_vectors, loaded_docs, top_k=5))[0]['doc']['id']
    print(f'{domain:<12} {query[:50]:<50} | {sparse_top:<6} | {dense_top:<6} | {hybrid_top:<6} | {reranked_top}')


sparse_top1  = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_sparse(q, bm25, loaded_docs, top_k=5))
dense_top1   = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_dense(q, embedder, dense_vectors, loaded_docs, top_k=5))
hybrid_top1  = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_hybrid(q, bm25, embedder, dense_vectors, loaded_docs, top_k=5))
rerank_top1  = top1_accuracy_for_queries(
    loaded_labeled_eval, lambda q: rerank_like_cross_encoder(q, rank_hybrid(q, bm25, embedder, dense_vectors, loaded_docs, top_k=5)))

print(f'\nTop-1 accuracy over {len(loaded_labeled_eval)}-query cross-domain eval set')
print(f'  Sparse     : {sparse_top1:.2f}')
print(f'  Dense      : {dense_top1:.2f}')
print(f'  Hybrid     : {hybrid_top1:.2f}')
print(f'  Hybrid+RR  : {rerank_top1:.2f}')

The comparison shows the shortlist quality before reranking. In this lab, the reranker is the practical last step because it can promote the most query-aligned candidate after dense and sparse retrieval have already done the cheap work.

Hybrid search helps when sparse and dense retrieval fail for different reasons. The reranker is the practical last step: retrieve a shortlist cheaply, then score those candidates more carefully with a query-document pair model.

## Retrieval quality metrics

These metrics answer a practical question: is the retriever actually finding the right chunks?

- **Recall@k** — how many of the relevant items were found in the top-k results (coverage).
- **Precision@k** — what fraction of the top-k results are actually relevant (cleanliness).
- **MRR** — mean reciprocal rank; rewards retrievers that put the first relevant hit near rank 1.
- **NDCG@k** — normalised discounted cumulative gain; supports graded relevance, so a grade-2 item at rank 1 is worth more than a grade-1 item at rank 3.

**Domain relevance:**
- A **finance** QA bot failing MRR means analysts get the wrong document first — a costly mistake when the answer drives a trade decision.
- An **edtech** search engine with low recall@5 means students can't find the right lesson materials even though they exist in the corpus.
- A **clinical** decision-support system with low precision@3 surfaces irrelevant treatment guidelines alongside the correct one, adding cognitive load for clinicians.

The labeled eval set below covers all of these domains so the metric comparison is grounded across contexts.

In [ ]:
# ── Metric table across all rankers ──────────────────────────────────────────
metric_table = [
    ('Sparse',    evaluate_ranker(loaded_labeled_eval, lambda q: rank_sparse(q, bm25, loaded_docs, top_k=5),  k=5)),
    ('Dense',     evaluate_ranker(loaded_labeled_eval, lambda q: rank_dense(q, embedder, dense_vectors, loaded_docs, top_k=5),  k=5)),
    ('Hybrid',    evaluate_ranker(loaded_labeled_eval, lambda q: rank_hybrid(q, bm25, embedder, dense_vectors, loaded_docs, top_k=5), k=5)),
    ('Hybrid+RR', evaluate_ranker(loaded_labeled_eval,
                    lambda q: rerank_like_cross_encoder(q, rank_hybrid(q, bm25, embedder, dense_vectors, loaded_docs, top_k=5)),        k=5)),
]

print(f'Metrics over {len(loaded_labeled_eval)}-query cross-domain eval set  (k=5)')
print(f'{"Method":<12} | Recall@5 | Precision@5 | MRR  | NDCG@5')
print('-' * 55)
for name, scores in metric_table:
    print(f"{name:<12} | {scores['recall']:.2f}     | {scores['precision']:.2f}        | "
          f"{scores['mrr']:.2f} | {scores['ndcg']:.2f}")

# ── Single-query walkthrough (finance example) ───────────────────────────────
sample_row = next(r for r in loaded_labeled_eval if r['metadata_filter'].get('category') == 'finance')
sample_ranked = rank_hybrid(sample_row['query'], bm25, embedder, dense_vectors, loaded_docs, top_k=5)
sample_ids    = [item['doc']['id'] for item in sample_ranked]
print(f'\nSample query  : {sample_row["query"]}')
print(f'Ranked ids    : {sample_ids}')
print(f'Recall@3      : {recall_at_k(sample_ids,    sample_row["relevance"], 3):.2f}')
print(f'Precision@3   : {precision_at_k(sample_ids, sample_row["relevance"], 3):.2f}')
print(f'MRR           : {reciprocal_rank(sample_ids, sample_row["relevance"]):.2f}')
print(f'NDCG@3        : {ndcg_at_k(sample_ids,       sample_row["relevance"], 3):.2f}')

The metric table turns the ranking behavior into numbers: recall shows coverage, precision shows cleanliness, MRR shows first-hit rank, and NDCG shows whether the best item is being pushed toward the top.